# WikiTrend Silver Inspection

This notebook inspects trusted Silver Parquet data, schemas, partitions, quality fields, and targeted DuckDB queries.

## 1. Project setup

Run this notebook from the repository root or from the `notebooks` directory.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.dataset as ds
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_DIR = PROJECT_ROOT / 'data' / 'silver' / 'pageviews'

print(f'Project root: {PROJECT_ROOT}')
print(f'Silver directory exists: {SILVER_DIR.exists()}')


## 2. Silver schema and bounded sample

Silver is partitioned by `date`, `hour`, canonical `project`, and `access_mode`. Use filters when inspecting a specific slice.

In [ ]:
silver = ds.dataset(SILVER_DIR, format='parquet', partitioning='hive')
print(silver.schema)
print(f'Parquet files: {len(silver.files):,}')
print(f'Rows: {silver.count_rows():,}')

In [ ]:
silver_columns = [
    'date', 'hour', 'source_project', 'project', 'language', 'project_family', 'access_mode',
    'page_title', 'normalized_title', 'normalization_status',
    'view_count', 'response_size', 'source_file'
]
silver_filter = (
    (ds.field('date') == '2026-08-01')
    & (ds.field('hour') == 0)
    & (ds.field('project') == 'en')
    & (ds.field('access_mode') == 'mobile')
)

print(f'Filtered rows: {silver.count_rows(filter=silver_filter):,}')
display(silver.head(10, columns=silver_columns, filter=silver_filter).to_pandas())

## 3. Silver quality EDA

Silver should contain only structurally valid parsed records. These checks verify that assumption and identify valid titles requiring a later business decision.

### Silver missing and invalid values

Silver should contain only valid parsed records, but these checks verify that assumption and identify valid titles that are candidates for later trend filtering.

In [ ]:
silver_glob = (SILVER_DIR / '**' / '*.parquet').as_posix()
symbol_only_pattern = r'^[^\p{L}\p{N}]+$'
con = duckdb.connect()
silver_eda = con.sql(f"""
    SELECT
        count(*) AS total_rows,
        sum(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS missing_date,
        sum(CASE WHEN hour IS NULL THEN 1 ELSE 0 END) AS missing_hour,
        sum(CASE WHEN hour IS NOT NULL AND (hour < 0 OR hour > 23) THEN 1 ELSE 0 END) AS invalid_hour,
        sum(CASE WHEN source_project IS NULL OR trim(source_project) = '' THEN 1 ELSE 0 END) AS missing_source_project,
        sum(CASE WHEN project IS NULL OR trim(project) = '' THEN 1 ELSE 0 END) AS missing_project,
        sum(CASE WHEN project_family = 'wikipedia' AND (language IS NULL OR trim(language) = '') THEN 1 ELSE 0 END) AS missing_wikipedia_language,
        sum(CASE WHEN project_family IS NULL OR trim(project_family) = '' THEN 1 ELSE 0 END) AS missing_project_family,
        sum(CASE WHEN access_mode NOT IN ('desktop', 'mobile') OR access_mode IS NULL THEN 1 ELSE 0 END) AS invalid_access_mode,
        sum(CASE WHEN page_title IS NULL OR trim(page_title) = '' THEN 1 ELSE 0 END) AS missing_page_title,
        sum(CASE WHEN normalized_title IS NULL OR trim(normalized_title) = '' THEN 1 ELSE 0 END) AS missing_normalized_title,
        sum(CASE WHEN normalized_title IS NOT NULL AND trim(normalized_title) <> ''
                 AND regexp_matches(trim(normalized_title), '{symbol_only_pattern}')
                 THEN 1 ELSE 0 END) AS symbol_only_titles,
        sum(CASE WHEN view_count IS NULL OR view_count < 0 THEN 1 ELSE 0 END) AS invalid_view_count,
        sum(CASE WHEN response_size IS NULL OR response_size < 0 THEN 1 ELSE 0 END) AS invalid_response_size,
        sum(CASE WHEN source_file IS NULL OR trim(source_file) = '' THEN 1 ELSE 0 END) AS missing_source_file
    FROM read_parquet('{silver_glob}', hive_partitioning=true)
""").df()
display(silver_eda.T.rename(columns={0: 'count'}))

### Inspect suspicious title values

These are valid Silver rows requiring a business decision, not automatic parser quarantine.

In [ ]:
suspicious_titles = con.sql(f"""
    SELECT source_project, project, access_mode, page_title, normalized_title, view_count, date, hour
    FROM read_parquet('{silver_glob}', hive_partitioning=true)
    WHERE normalized_title IS NULL
       OR trim(normalized_title) = ''
       OR regexp_matches(trim(normalized_title), '{symbol_only_pattern}')
    ORDER BY view_count DESC
    LIMIT 50
""").df()
display(suspicious_titles)

In [ ]:
rows_by_project = con.sql(f"""
    SELECT source_project, project, access_mode, language, project_family, count(*) AS rows, sum(view_count) AS total_views
    FROM read_parquet('{silver_glob}', hive_partitioning=true)
    GROUP BY source_project, project, access_mode, language, project_family
    ORDER BY total_views DESC
""").df()
display(rows_by_project)

coverage = con.sql(f"""
    SELECT date, hour, count(*) AS rows, count(DISTINCT source_project) AS source_projects, sum(view_count) AS total_views
    FROM read_parquet('{silver_glob}', hive_partitioning=true)
    GROUP BY date, hour
    ORDER BY date, hour
""").df()
display(coverage.head(10))
print(f'Date-hour windows in Silver: {len(coverage):,}')

## 4. Silver SQL inspection with DuckDB

DuckDB is useful for targeted aggregates and top-page inspection without materializing the full table.

In [ ]:
silver_glob = (SILVER_DIR / '**' / '*.parquet').as_posix()
con = duckdb.connect()

top_pages = con.sql(f"""
    SELECT normalized_title, view_count, source_project, project, access_mode
    FROM read_parquet('{silver_glob}', hive_partitioning=true)
    WHERE date = '2026-08-01'
      AND hour = 0
      AND project = 'en'
      AND access_mode = 'mobile'
    ORDER BY view_count DESC
    LIMIT 25
""")
display(top_pages)

In [ ]:
quality = con.sql(f"""
    SELECT
        count(*) AS rows,
        count(DISTINCT date) AS dates,
        count(DISTINCT hour) AS hours,
        count(DISTINCT project) AS projects,
        count(DISTINCT source_project) AS source_projects,
        count(DISTINCT access_mode) AS access_modes,
        sum(CASE WHEN view_count IS NULL OR view_count < 0 THEN 1 ELSE 0 END) AS invalid_views,
        sum(CASE WHEN response_size IS NULL OR response_size < 0 THEN 1 ELSE 0 END) AS invalid_response_sizes,
        sum(CASE WHEN page_title IS NULL OR normalized_title IS NULL THEN 1 ELSE 0 END) AS invalid_titles
    FROM read_parquet('{silver_glob}', hive_partitioning=true)
""")
display(quality)
con.close()

## Inspection workflow

1. Inspect the Silver schema and Parquet file manifest.
2. Use partition filters to sample a date, hour, and project.
3. Run missing-value, invalid-value, title, project, and coverage checks.
4. Use DuckDB for targeted aggregates and top-page queries.
5. Run `python scripts/validate_silver.py` for the full validation report.